# Phase 4 Agent Scratch

This notebook is for testing the agent idea before moving anything into `.py` files.

Goal: take one chest X-ray image, predict findings, retrieve KB evidence, draft a short report, then check if the draft is grounded.

## What I Am Testing Here

Phase 4 connects the tools from earlier phases:

- Phase 2: vision model predicts possible findings from the image
- Phase 3: retriever finds short medical evidence passages
- Phase 4: agent uses both to write a careful draft

The agent should not act like a doctor. It should say what the model found, use retrieved evidence, and stay careful when confidence is low.

## Step 0: Imports And Paths

In [90]:
from pathlib import Path
import json
import os

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(PROJECT_ROOT)

E:\Project\RadScribe


## Step 1: Load Existing Tools

These are the tools already built in Phase 2 and Phase 3.

In [91]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.models.predict import predict_findings
from src.rag.retrieve import retrieve

print("tools loaded")

tools loaded


## Step 2: Choose Test Case

Use one variable to switch between the three cases the mentor asked for.

In [92]:
manifest = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "manifest.parquet")
test_df = manifest[manifest["split"] == "test"].copy()

DISEASE_LABELS = [
    "Cardiomegaly",
    "Atelectasis",
    "Consolidation / Pneumonia",
    "Pleural Effusion",
    "Edema",
    "Pneumothorax",
]


def label_list(labels):
    return list(labels)


def has_label(labels, target):
    return target in label_list(labels)


def is_other_only(labels):
    labels = label_list(labels)
    return "Other" in labels and not any(label in labels for label in DISEASE_LABELS)


def is_no_finding_only(labels):
    return label_list(labels) == ["No Finding"]


case_options = {
    "clear_positive": test_df[test_df["labels"].apply(lambda labels: has_label(labels, "Cardiomegaly"))].sample(1, random_state=2).iloc[0],
    "normal": test_df[test_df["labels"].apply(is_no_finding_only)].sample(1, random_state=3).iloc[0],
    "other_low_confidence": test_df[test_df["labels"].apply(is_other_only)].sample(1, random_state=4).iloc[0],
}

# Change this to: "clear_positive", "normal", or "other_low_confidence".
CASE_NAME = "clear_positive"

row = case_options[CASE_NAME]
image_path = PROJECT_ROOT / Path(str(row["image_path"]).replace("\\", "/"))

print("case:", CASE_NAME)
print("study_id:", row["study_id"])
print("labels:", row["labels"])
print("image:", image_path)

case: clear_positive
study_id: 797
labels: ['Cardiomegaly']
image: E:\Project\RadScribe\data\processed\images_224\797_IM-2332-1001.dcm.png


## Step 3: Vision Prediction

This calls the Phase 2 model and keeps only findings above a simple threshold.

In [93]:
BORDERLINE_VISION_THRESHOLD = 0.50
MAIN_VISION_THRESHOLD = 0.70

vision_probs = predict_findings(image_path)
vision_df = (
    pd.DataFrame(
        [{"finding": finding, "probability": prob} for finding, prob in vision_probs.items()]
    )
    .sort_values("probability", ascending=False)
    .reset_index(drop=True)
)

main_df = vision_df[vision_df["probability"] >= MAIN_VISION_THRESHOLD].copy()
borderline_df = vision_df[
    (vision_df["probability"] >= BORDERLINE_VISION_THRESHOLD)
    & (vision_df["probability"] < MAIN_VISION_THRESHOLD)
].copy()

if main_df.empty and borderline_df.empty:
    vision_status = "no_finding_above_threshold"
elif not main_df.empty:
    vision_status = "high_confidence"
else:
    vision_status = "borderline"

main_findings = main_df["finding"].tolist()
borderline_findings = borderline_df["finding"].tolist()
retrieval_findings = main_findings + borderline_findings

display(vision_df)
print("main findings:", main_findings)
print("borderline findings:", borderline_findings)
print("vision status:", vision_status)

,finding,probability
0,Cardiomegaly,0.956632
1,Edema,0.932232
2,Consolidation / Pneumonia,0.615842
3,Atelectasis,0.455839
4,Pleural Effusion,0.325371
5,Pneumothorax,0.064075


main findings: ['Cardiomegaly', 'Edema']
borderline findings: ['Consolidation / Pneumonia']
vision status: high_confidence


## Step 4: Retrieve Evidence

Only retrieve evidence if the vision model found something above the confidence threshold.

In [94]:
RETRIEVAL_K = 3
MIN_GROUNDING_SCORE = 0.38

evidence_rows = []

if retrieval_findings:
    for finding in retrieval_findings:
        prob = float(vision_df[vision_df["finding"] == finding]["probability"].iloc[0])
        confidence_group = "main" if finding in main_findings else "borderline"
        query = f"chest xray finding {finding} appearance"
        for hit in retrieve(query, k=RETRIEVAL_K):
            evidence_rows.append(
                {
                    "query_finding": finding,
                    "vision_probability": prob,
                    "confidence_group": confidence_group,
                    **hit,
                }
            )

evidence_df = pd.DataFrame(evidence_rows)

if evidence_df.empty:
    display(pd.DataFrame(columns=["query_finding", "rank", "finding", "score", "chunk_id", "text"]))
else:
    display(evidence_df[["query_finding", "confidence_group", "vision_probability", "rank", "finding", "score", "chunk_id", "text"]])

best_score = evidence_df["score"].max() if len(evidence_df) else 0.0
retrieval_ok = bool(best_score >= MIN_GROUNDING_SCORE) if retrieval_findings else False
can_draft = bool(main_findings) and retrieval_ok

print("best retrieval score:", round(best_score, 3))
print("retrieval ok:", retrieval_ok)
print("can draft:", can_draft)

,query_finding,confidence_group,vision_probability,rank,finding,score,chunk_id,text
0,Cardiomegaly,main,0.956632,1,Cardiomegaly,0.747722,cardiomegaly_001,The finding matters because an enlarged cardia...
1,Cardiomegaly,main,0.956632,2,Cardiomegaly,0.709868,cardiomegaly_002,A common measurement is the cardiothoracic rat...
2,Cardiomegaly,main,0.956632,3,Cardiomegaly,0.705224,cardiomegaly_000,Cardiomegaly means the heart appears enlarged....
3,Edema,main,0.932232,1,Edema,0.685271,edema_001,"Pulmonary edema can cause shortness of breath,..."
4,Edema,main,0.932232,2,Pleural Effusion,0.624226,pleural_effusion_002,"On chest X-ray, pleural effusion is usually co..."
5,Edema,main,0.932232,3,Cardiomegaly,0.590225,cardiomegaly_001,The finding matters because an enlarged cardia...
6,Consolidation / Pneumonia,borderline,0.615842,1,Consolidation / Pneumonia,0.764533,consolidation_pneumonia_001,Consolidation is a radiographic pattern where ...
7,Consolidation / Pneumonia,borderline,0.615842,2,Consolidation / Pneumonia,0.709804,consolidation_pneumonia_002,It may look denser or whiter than normal aerat...
8,Consolidation / Pneumonia,borderline,0.615842,3,Consolidation / Pneumonia,0.616031,consolidation_pneumonia_000,Pneumonia is an infection of the lungs. It can...


best retrieval score: 0.765
retrieval ok: True
can draft: True


## Step 5: Agent State

Keep the agent state small and easy to inspect. This makes debugging much easier.

In [95]:
agent_state = {
    "image_path": str(image_path),
    "study_id": str(row["study_id"]),
    "case_name": CASE_NAME,
    "vision_predictions": vision_df.to_dict("records"),
    "main_findings": main_findings,
    "borderline_findings": borderline_findings,
    "retrieval_findings": retrieval_findings,
    "vision_status": vision_status,
    "evidence": evidence_df.to_dict("records"),
    "best_retrieval_score": float(best_score),
    "retrieval_ok": retrieval_ok,
    "can_draft": can_draft,
}

agent_state.keys()

dict_keys(['image_path', 'study_id', 'case_name', 'vision_predictions', 'main_findings', 'borderline_findings', 'retrieval_findings', 'vision_status', 'evidence', 'best_retrieval_score', 'retrieval_ok', 'can_draft'])

## Step 6: Draft Prompt

This prompt is strict on purpose. The model should only use the vision output and retrieved evidence.

In [96]:
def short_evidence_block(evidence: pd.DataFrame, max_rows: int = 5) -> str:
    if evidence.empty:
        return "No retrieved evidence because no finding passed the vision threshold."

    lines = []
    for item in evidence.head(max_rows).to_dict("records"):
        lines.append(
            f"- [{item['chunk_id']}] finding={item['finding']} score={item['score']:.3f}: {item['text']}"
        )
    return "\n".join(lines)


def make_draft_prompt(state: dict) -> str:
    evidence = pd.DataFrame(state["evidence"])
    main = ", ".join(state["main_findings"]) if state["main_findings"] else "None"
    borderline = ", ".join(state["borderline_findings"]) if state["borderline_findings"] else "None"
    evidence_text = short_evidence_block(evidence)
    vision_rows = []
    for item in state["vision_predictions"]:
        vision_rows.append(f"- {item['finding']}: {item['probability']:.3f}")
    vision_text = "\n".join(vision_rows)

    return f"""
Draft a concise chest X-ray report from the information below.

Use the vision findings as model output, not as a final diagnosis.
Use the retrieved passages only for grounding.
Do not add findings that are not listed or supported.
Use careful wording like "possible" or "the model suggests".
Do not write as if this is a confirmed radiologist diagnosis.
Main findings are the only findings allowed in the main impression.
Borderline findings may be mentioned only as lower-confidence observations, not as the main conclusion.
If there are no main findings, do not draft a disease-specific report.

All vision probabilities:
{vision_text}

Vision status:
{state['vision_status']}

Main model findings, probability >= 0.70:
{main}

Borderline model findings, probability 0.50-0.70:
{borderline}

Best retrieval score:
{state['best_retrieval_score']:.3f}

Retrieved evidence:
{evidence_text}

Write a professional draft with these sections:
Findings: short paragraph.
Impression: 1-2 short bullets.
Evidence: cite chunk ids in parentheses when evidence exists.
""".strip()


draft_prompt = make_draft_prompt(agent_state)
print(draft_prompt)

Draft a concise chest X-ray report from the information below.

Use the vision findings as model output, not as a final diagnosis.
Use the retrieved passages only for grounding.
Do not add findings that are not listed or supported.
Use careful wording like "possible" or "the model suggests".
Do not write as if this is a confirmed radiologist diagnosis.
Main findings are the only findings allowed in the main impression.
Borderline findings may be mentioned only as lower-confidence observations, not as the main conclusion.
If there are no main findings, do not draft a disease-specific report.

All vision probabilities:
- Cardiomegaly: 0.957
- Edema: 0.932
- Consolidation / Pneumonia: 0.616
- Atelectasis: 0.456
- Pleural Effusion: 0.325
- Pneumothorax: 0.064

Vision status:
high_confidence

Main model findings, probability >= 0.70:
Cardiomegaly, Edema

Borderline model findings, probability 0.50-0.70:
Consolidation / Pneumonia

Best retrieval score:
0.765

Retrieved evidence:
- [cardiomeg

## Step 7: Draft With OpenAI

This is the first LLM call. If retrieval is below the threshold, skip drafting and return a low-confidence message.

In [97]:
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env")
except ModuleNotFoundError:
    pass

from openai import OpenAI

DRAFT_MODEL = "gpt-4o-mini"

if not agent_state["main_findings"]:
    if agent_state["borderline_findings"]:
        draft_report = "Only borderline model findings were present. No main disease-specific report was generated."
    else:
        draft_report = "No finding above the model confidence threshold. No disease-specific draft was generated."
elif not agent_state["retrieval_ok"]:
    draft_report = "Low confidence: the model selected a finding, but retrieval did not find enough supporting evidence."
else:
    client = OpenAI()
    response = client.chat.completions.create(
        model=DRAFT_MODEL,
        messages=[
            {"role": "system", "content": "Write concise, evidence-grounded chest X-ray draft text. Never state a finding as confirmed. Use wording like 'model suggests', 'possible', or 'may represent'."},
            {"role": "user", "content": draft_prompt},
        ],
        temperature=0.2,
    )
    draft_report = response.choices[0].message.content

print(draft_report)

**Findings:**  
The chest X-ray demonstrates possible cardiomegaly, indicated by an enlarged cardiac silhouette, which may suggest underlying cardiac strain or dysfunction. Additionally, there is a high probability of pulmonary edema, characterized by potential increased vascular markings and other associated patterns. There is a borderline observation of consolidation or pneumonia, but this finding is less certain.

**Impression:**  
- Possible cardiomegaly and pulmonary edema.  
- Borderline consolidation or pneumonia may represent an additional consideration.

**Evidence:**  
- Cardiomegaly (cardiomegaly_001, cardiomegaly_002, cardiomegaly_000)  
- Edema (edema_001)  
- Consolidation / Pneumonia (noted as borderline)


## Step 8: Critic Check

The critic should check if the draft is supported by the retrieved evidence. It should be strict.

In [98]:
def make_critic_prompt(draft: str, state: dict) -> str:
    evidence = pd.DataFrame(state["evidence"])
    evidence_text = short_evidence_block(evidence, max_rows=8)

    return f"""
Check this draft against the evidence.

Return raw JSON only. Do not wrap it in markdown or code fences.

Use these keys:
- supported: true or false
- missing_evidence: list of claims not supported
- safety_note: short note

Draft:
{draft}

Evidence:
{evidence_text}
""".strip()


critic_prompt = make_critic_prompt(draft_report, agent_state)
print(critic_prompt)

Check this draft against the evidence.

Return raw JSON only. Do not wrap it in markdown or code fences.

Use these keys:
- supported: true or false
- missing_evidence: list of claims not supported
- safety_note: short note

Draft:
**Findings:**  
The chest X-ray demonstrates possible cardiomegaly, indicated by an enlarged cardiac silhouette, which may suggest underlying cardiac strain or dysfunction. Additionally, there is a high probability of pulmonary edema, characterized by potential increased vascular markings and other associated patterns. There is a borderline observation of consolidation or pneumonia, but this finding is less certain.

**Impression:**  
- Possible cardiomegaly and pulmonary edema.  
- Borderline consolidation or pneumonia may represent an additional consideration.

**Evidence:**  
- Cardiomegaly (cardiomegaly_001, cardiomegaly_002, cardiomegaly_000)  
- Edema (edema_001)  
- Consolidation / Pneumonia (noted as borderline)

Evidence:
- [cardiomegaly_001] findin

In [99]:
client = OpenAI()
critic_response = client.chat.completions.create(
    model=DRAFT_MODEL,
    messages=[
        {"role": "system", "content": "You are a strict evidence checker. Return JSON only."},
        {"role": "user", "content": critic_prompt},
    ],
    temperature=0,
)

critic_text = critic_response.choices[0].message.content
print(critic_text)

{
  "supported": true,
  "missing_evidence": [],
  "safety_note": "The findings are consistent with the evidence provided."
}


## Step 9: LangGraph Shape

This is the graph structure we want later in real code.

For this scratch notebook, keep it simple first. After the plain functions work, move them into LangGraph nodes.

In [100]:
# Planned graph:
#
# image_path
#   -> vision_node
#   -> retrieve_node
#   -> draft_node
#   -> critic_node
#   -> final_node
#
# Keep this notebook focused on proving the behavior first.
# Then port the clean version into src/agent/.

print("Graph plan written. Build LangGraph after this basic flow is approved.")

Graph plan written. Build LangGraph after this basic flow is approved.


## Step 10: Three Case Trace

Run this cell before sending the notebook. It keeps the three mentor cases visible in one output.

In [101]:
def run_case_trace(case_name: str) -> dict:
    case_row = case_options[case_name]
    case_image_path = PROJECT_ROOT / Path(str(case_row["image_path"]).replace("\\", "/"))

    probs = predict_findings(case_image_path)
    probs_df = (
        pd.DataFrame([{"finding": finding, "probability": prob} for finding, prob in probs.items()])
        .sort_values("probability", ascending=False)
        .reset_index(drop=True)
    )

    case_main_df = probs_df[probs_df["probability"] >= MAIN_VISION_THRESHOLD].copy()
    case_borderline_df = probs_df[
        (probs_df["probability"] >= BORDERLINE_VISION_THRESHOLD)
        & (probs_df["probability"] < MAIN_VISION_THRESHOLD)
    ].copy()

    if case_main_df.empty and case_borderline_df.empty:
        case_status = "no_finding_above_threshold"
    elif not case_main_df.empty:
        case_status = "high_confidence"
    else:
        case_status = "borderline"

    case_main_findings = case_main_df["finding"].tolist()
    case_borderline_findings = case_borderline_df["finding"].tolist()
    case_retrieval_findings = case_main_findings + case_borderline_findings

    case_evidence_rows = []
    for finding in case_retrieval_findings:
        prob = float(probs_df[probs_df["finding"] == finding]["probability"].iloc[0])
        group = "main" if finding in case_main_findings else "borderline"
        query = f"chest xray finding {finding} appearance"
        for hit in retrieve(query, k=RETRIEVAL_K):
            case_evidence_rows.append(
                {
                    "query_finding": finding,
                    "vision_probability": prob,
                    "confidence_group": group,
                    **hit,
                }
            )

    case_evidence_df = pd.DataFrame(case_evidence_rows)
    case_best_score = case_evidence_df["score"].max() if len(case_evidence_df) else 0.0
    case_retrieval_ok = bool(case_best_score >= MIN_GROUNDING_SCORE) if case_retrieval_findings else False
    case_can_draft = bool(case_main_findings) and case_retrieval_ok

    case_state = {
        "image_path": str(case_image_path),
        "study_id": str(case_row["study_id"]),
        "case_name": case_name,
        "vision_predictions": probs_df.to_dict("records"),
        "main_findings": case_main_findings,
        "borderline_findings": case_borderline_findings,
        "retrieval_findings": case_retrieval_findings,
        "vision_status": case_status,
        "evidence": case_evidence_df.to_dict("records"),
        "best_retrieval_score": float(case_best_score),
        "retrieval_ok": case_retrieval_ok,
        "can_draft": case_can_draft,
    }

    if not case_state["main_findings"]:
        if case_state["borderline_findings"]:
            case_draft = "Only borderline model findings were present. No main disease-specific report was generated."
        else:
            case_draft = "No finding above the model confidence threshold. No disease-specific draft was generated."
    elif not case_state["retrieval_ok"]:
        case_draft = "Low confidence: the model selected a finding, but retrieval did not find enough supporting evidence."
    else:
        case_prompt = make_draft_prompt(case_state)
        response = client.chat.completions.create(
            model=DRAFT_MODEL,
            messages=[
                {"role": "system", "content": "Write concise, evidence-grounded chest X-ray draft text. Never state a finding as confirmed. Use wording like 'model suggests', 'possible', or 'may represent'."},
                {"role": "user", "content": case_prompt},
            ],
            temperature=0.2,
        )
        case_draft = response.choices[0].message.content

    return {
        "case": case_name,
        "study_id": str(case_row["study_id"]),
        "true_labels": list(case_row["labels"]),
        "top_probabilities": probs_df.head(3).to_dict("records"),
        "main_findings": case_main_findings,
        "borderline_findings": case_borderline_findings,
        "vision_status": case_status,
        "best_retrieval_score": round(float(case_best_score), 3),
        "retrieval_ok": case_retrieval_ok,
        "can_draft": case_can_draft,
        "draft": case_draft,
    }


case_traces = [run_case_trace(name) for name in ["clear_positive", "normal", "other_low_confidence"]]

for trace in case_traces:
    print("=" * 80)
    print("case:", trace["case"])
    print("study_id:", trace["study_id"])
    print("true labels:", trace["true_labels"])
    print("top probabilities:", trace["top_probabilities"])
    print("main findings:", trace["main_findings"])
    print("borderline findings:", trace["borderline_findings"])
    print("vision status:", trace["vision_status"])
    print("best retrieval score:", trace["best_retrieval_score"])
    print("retrieval ok:", trace["retrieval_ok"])
    print("can draft:", trace["can_draft"])
    print("draft:")
    print(trace["draft"])

pd.DataFrame(case_traces)[[
    "case",
    "study_id",
    "true_labels",
    "main_findings",
    "borderline_findings",
    "vision_status",
    "best_retrieval_score",
    "retrieval_ok",
    "can_draft",
]]

case: clear_positive
study_id: 797
true labels: ['Cardiomegaly']
top probabilities: [{'finding': 'Cardiomegaly', 'probability': 0.9566320776939392}, {'finding': 'Edema', 'probability': 0.9322324395179749}, {'finding': 'Consolidation / Pneumonia', 'probability': 0.6158415675163269}]
main findings: ['Cardiomegaly', 'Edema']
borderline findings: ['Consolidation / Pneumonia']
vision status: high_confidence
best retrieval score: 0.765
retrieval ok: True
can draft: True
draft:
**Findings:**  
The chest X-ray demonstrates possible cardiomegaly, indicated by an enlarged cardiac silhouette, which may suggest underlying cardiac strain or dysfunction. Additionally, there is a high probability of pulmonary edema, which could manifest as increased vascular markings and other associated patterns. There is a borderline likelihood of consolidation or pneumonia.

**Impression:**  
- Possible cardiomegaly and edema.  
- Borderline findings may represent consolidation or pneumonia.

**Evidence:**  
- Car

,case,study_id,true_labels,main_findings,borderline_findings,vision_status,best_retrieval_score,retrieval_ok,can_draft
0,clear_positive,797,[Cardiomegaly],"[Cardiomegaly, Edema]",[Consolidation / Pneumonia],high_confidence,0.765,True,True
1,normal,3528,[No Finding],[],[],no_finding_above_threshold,0.000,False,False
2,other_low_confidence,1131,[Other],[],[],no_finding_above_threshold,0.000,False,False


## What To Show Mentor

After running this notebook, send:

- selected image path and true labels
- vision probabilities
- main findings and borderline findings
- top retrieved evidence with scores
- draft report
- critic JSON

Do not port to `.py` until the basic behavior is approved.